In [18]:
import os
import mne
import numpy as np
import pandas as pd
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from tqdm.notebook import tqdm

## Data  Loading

In [2]:
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")
df_filtered = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]
df_skill = df_skill[["Participant", "SkillScore"]]
df_skill = df_skill.drop_duplicates()
# assign skill level
quantile = df_skill["SkillScore"].quantile([0.33, 0.66])
lower = quantile[0.33]
upper = quantile[0.66]

df_skill['SkillLevel'] = np.select([df_skill['SkillScore'] < lower, (df_skill['SkillScore'] >= lower) & (df_skill['SkillScore'] <= upper), df_skill['SkillScore'] > upper],
                                 ['Novice', 'Intermediate', 'Expert'],
                                 default='Intermediate')
df_skilled = df_skill.copy()
df_filtered = pd.merge(df_filtered, df_skilled[['Participant', 'SkillLevel']], on='Participant', how='left')
df_filtered = df_filtered[["Participant", "Algorithm", "SkillLevel", "EEG", "CrossEEG"]]
df_filtered

,Participant,Algorithm,SkillScore,EEG,CrossEEG
0,1,IsPrime,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1,1,SiebDesEratosthenes,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
2,1,IsAnagram,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
3,1,RemoveDoubleChar,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
4,1,BinToDecimal,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
...,...,...,...,...,...
1067,71,DumpSorting,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1068,71,BinomialCoefficient,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1069,71,IsAnagram,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1070,71,ArrayAverage,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...


In [5]:
df_skill_map = dict(zip(df_filtered["Participant"], df_filtered["SkillLevel"]))
df_skill_map

{1: 'Intermediate',
 2: 'Expert',
 3: 'Intermediate',
 4: 'Expert',
 5: 'Intermediate',
 6: 'Intermediate',
 7: 'Expert',
 10: 'Expert',
 11: 'Novice',
 12: 'Intermediate',
 13: 'Expert',
 14: 'Intermediate',
 18: 'Novice',
 22: 'Intermediate',
 24: 'Novice',
 25: 'Intermediate',
 28: 'Novice',
 35: 'Expert',
 36: 'Expert',
 37: 'Intermediate',
 38: 'Expert',
 41: 'Intermediate',
 42: 'Intermediate',
 49: 'Novice',
 50: 'Novice',
 55: 'Novice',
 58: 'Novice',
 59: 'Novice',
 60: 'Expert',
 61: 'Novice',
 62: 'Expert',
 63: 'Intermediate',
 66: 'Expert',
 67: 'Novice',
 68: 'Expert',
 70: 'Novice',
 71: 'Expert'}

In [6]:
def read_eeg_raw(eeg_path):
    raw = mne.io.read_raw_fif(eeg_path, preload= True)
    return raw

In [25]:
def padding_arrays(eeg_data):
    max_time_points = max(data.shape[1] for data in eeg_data)
    for i, data in enumerate(eeg_data):
        time_points = data.shape[1]
        if time_points < max_time_points:
            eeg_data[i] = np.pad(data,((0,0), (0, max_time_points - time_points)), mode = 'constant')
        elif time_points > max_time_points:
            eeg_data[i] = data[:, :max_time_points]
    return eeg_data

In [38]:
def get_csp_one_vs_rest(df_eeg, skill_map, skill_level):

    # Choose participant number of the selected skill level
    participants = [participant for participant, skill_lvl in skill_map.items() if skill_lvl == skill_level]

    eeg_data = []


    for participant in tqdm(participants, desc="Processing Participants"):
        eeg_data_participant = []
        #filter by skill levels

        eeg_path_participant = df_eeg[(df_eeg["Participant"]== participant)& (df_eeg["SkillLevel"]== skill_level)]

        for _, row in eeg_path_participant.iterrows():
            eeg_path = row["EEG"]
            raw = mne.io.read_raw_fif(eeg_path, preload= True)

            # Considered min. duration of the Participant's code comprehension as the time window
            time_window = 4

            # calculate EEG Signal duration
            eeg_duration = raw.n_times / raw.info['sfreq']

            # calculate midpoint of the signal
            midpoint = eeg_duration/2

            #Set time window's starting and end point to choose the middle part of the EEG signal
            tmin = midpoint - (time_window/2)
            tmax = midpoint + (time_window/2)
    
            eeg_data_participant.append(raw.copy().crop(tmin=tmin, tmax=tmax).pick_types(eeg=True).get_data())

#ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 2001 and the array at index 16 has size 2002
        

        eeg_data_participant_concat = np.concatenate(padding_arrays(eeg_data_participant), axis = 0)
        eeg_data.append(eeg_data_participant_concat)

    eeg_data_concat= np.concatenate(padding_arrays(eeg_data), axis= 0) 

    cov_matrix = np.cov(eeg_data_concat, rowvar = False)
    
    return cov_matrix

In [41]:
df_eeg = df_filtered.copy()
cov_matrices = {}
skill_levels = set(df_skill_map.values())


for skill_level in tqdm(skill_levels, desc= "Processing skill level"):
    cov_matrices[skill_level] = get_csp_one_vs_rest(df_eeg, df_skill_map, skill_level)

Processing skill level:   0%|          | 0/3 [00:00<?, ?it/s]

Processing Participants:   0%|          | 0/12 [00:00<?, ?it/s]

Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant11/DumpSortingcode_eeg_raw.fif...


    Range : 3749 ... 29489 =      7.498 ...    58.978 secs
Ready.
Reading 0 ... 25740  =      0.000 ...    51.480 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant11/BogoSortcode_eeg_raw.fif...
    Range : 71769 ... 157919 =    143.538 ...   315.838 secs
Ready.
Reading 0 ... 86150  =      0.000 ...   172.300 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant11/ArrayAveragecode_eeg_raw.fif...
    Range : 184639 ... 200859 =    369.278 ...   401.718 secs
Ready.
Reading 0 ... 16220  =      0.000 ...    32.440 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant11/IsPrimecode_eeg_raw.fif...
    Range : 224289 .

Processing Participants:   0%|          | 0/12 [00:00<?, ?it/s]

Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/IsPrimecode_eeg_raw.fif...
    Range : 6629 ... 12819 =     13.258 ...    25.638 secs
Ready.
Reading 0 ... 6190  =      0.000 ...    12.380 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/SiebDesEratosthenescode_eeg_raw.fif...
    Range : 31519 ... 107779 =     63.038 ...   215.558 secs
Ready.
Reading 0 ... 76260  =      0.000 ...   152.520 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/IsAnagramcode_eeg_raw.fif...
    Range : 133679 ... 188469 =    267.358 ...   376.938 secs
Ready.
Reading 0 ... 54790  =      0.000 ...   109.580 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw d

Processing Participants:   0%|          | 0/13 [00:00<?, ?it/s]

Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant02/HeightOfTreecode_eeg_raw.fif...
    Range : 4019 ... 24329 =      8.038 ...    48.658 secs
Ready.
Reading 0 ... 20310  =      0.000 ...    40.620 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant02/SignCheckercode_eeg_raw.fif...
    Range : 54889 ... 63759 =    109.778 ...   127.518 secs
Ready.
Reading 0 ... 8870  =      0.000 ...    17.740 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant02/Ackermancode_eeg_raw.fif...
    Range : 89309 ... 117779 =    178.618 ...   235.558 secs
Ready.
Reading 0 ... 28470  =      0.000 ...    56.940 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Opening raw data fi

In [47]:
df_skill_map

{1: 'Intermediate',
 2: 'Expert',
 3: 'Intermediate',
 4: 'Expert',
 5: 'Intermediate',
 6: 'Intermediate',
 7: 'Expert',
 10: 'Expert',
 11: 'Novice',
 12: 'Intermediate',
 13: 'Expert',
 14: 'Intermediate',
 18: 'Novice',
 22: 'Intermediate',
 24: 'Novice',
 25: 'Intermediate',
 28: 'Novice',
 35: 'Expert',
 36: 'Expert',
 37: 'Intermediate',
 38: 'Expert',
 41: 'Intermediate',
 42: 'Intermediate',
 49: 'Novice',
 50: 'Novice',
 55: 'Novice',
 58: 'Novice',
 59: 'Novice',
 60: 'Expert',
 61: 'Novice',
 62: 'Expert',
 63: 'Intermediate',
 66: 'Expert',
 67: 'Novice',
 68: 'Expert',
 70: 'Novice',
 71: 'Expert'}

In [57]:
# Combine covariance matrices and labels
cov_matrices_list = [cov_matrices[skill_level] for skill_level in skill_levels]
cov_matrices_list

AttributeError: 'list' object has no attribute 'shape'

In [68]:
# Initialize an empty list to store labels
labels_list = []
total_samples = 0
# Loop through participant IDs and their skill levels
for participant_id, skill_level in df_skill_map.items():
    # Get the number of EEG samples for this participant
    num_samples = len(df_skill_map[participant_id])
    total_samples += num_samples
    # Add the skill level label for this participant for each of their EEG samples
    labels_list.extend([skill_level] * num_samples)

# Check if the total number of samples matches the length of labels
if total_samples != len(labels_list):
    raise ValueError("Mismatch between number of samples and labels")

In [65]:
# Check if any of the covariance matrices or labels lists are empty
if not cov_matrices_list or not labels_list:
    raise ValueError("No data available for fitting CSP")

# Flatten the lists
cov_matrices_concatenated = np.concatenate(cov_matrices_list, axis=0)
labels_concatenated = np.array(labels_list)

In [67]:
n_components = 4

csp = CSP(n_components= n_components, log= True, reg= None)

# Fit CSP using covariance matrices and labels
csp.fit(cov_matrices_concatenated, labels_concatenated)

ValueError: X and y must have the same length.

In [ ]:

# Fit CSP using covariance matrices and labels
csp.fit(cov_matrices_concatenated, labels_concatenated)


## CSP

In [100]:
eeg_path = df_expert.iloc[0]["EEG"]
# read in eeg file
eeg_data = mne.io.read_raw_fif(eeg_path, preload=True, verbose='ERROR')

# get raw channel data and do mean average referencing
eeg_data_raw = eeg_data.get_data()
eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

# extract channel names of eeg_data
channel_names = list(eeg_data.to_data_frame().columns[1:])

# create mock events for cutting eeg data (number, len, id)
events = np.array([(0, 0, 1)])

# create temporal eeg raw for cutting data into epochs
tmp_raw = mne.io.RawArray(eeg_data_ref, eeg_data.info, verbose='ERROR')

# Considered min. duration of the Participant's code comprehension as the time window
time_window = 4

# calculate EEG Signal duration
eeg_duration = tmp_raw.n_times / tmp_raw.info['sfreq']

# calculate midpoint of the signal
midpoint = eeg_duration/2

#Set time window's starting and end point to choose the middle part of the EEG signal
tmin = midpoint - (time_window/2)
tmax = midpoint + (time_window/2)

# create epochs which have a duration second window and operate on event id 1
epochs = mne.Epochs(tmp_raw, events, event_id=1, tmin=tmin, tmax=tmax,baseline=None, preload=True, verbose='ERROR')

label = np.array([0])
label

array([0])

In [101]:

#apply CSP
csp = CSP(n_components=4, reg= None, log= True, norm_trace=False)
pipe = Pipeline([('CSP', csp), ('LDA', LDA())])
X = epochs.get_data()
y = label


In [102]:
X.shape, y.shape

((1, 64, 2001), (1,))

In [103]:
pipe.fit(X,y)

ValueError: n_classes must be >= 2.

In [11]:
eeg_expert.save('eeg_expert.edf')

OSError: The filename (c:\Users\Mahima Acharya\Documents\BCI\Data\Master-Thesis\Brain Regions\eeg_expert.edf) for file type raw must end with .fif or .fif.gz

In [22]:
X_expert = eeg_expert.get_data().T
y_expert = np.zeros(X_expert.shape[0]) #Experts are labelled as 0s
y_expert

array([0., 0., 0., ..., 0., 0., 0.])

In [32]:
X_intermediate = eeg_intermediate.get_data().T
y_intermediate = np.ones(X_intermediate.shape[0]) #intermediates are labelled as 1s
y_intermediate

array([1., 1., 1., ..., 1., 1., 1.])

In [33]:
X_novice = eeg_novice .get_data().T
y_novice  = np.ones(X_novice .shape[0])*2 #novices  are labelled as 2s
y_novice

array([2., 2., 2., ..., 2., 2., 2.])

In [34]:
X_test = eeg_test.get_data().T

In [35]:
X_train = np.concatenate([X_expert, X_intermediate, X_novice])
y_train = np.concatenate([y_expert, y_intermediate, y_novice])

In [36]:
#CSP

n_components = 4
csp = CSP(n_components, reg=None, log=None, norm_trace=False)
X_train_csp = csp.fit_transform(X_train, y_train)

ValueError: X must have at least 3 dimensions.